# Imports and Load

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
import xgboost as xgb
import mlflow
import re

mlflow.set_tracking_uri("http://localhost:5000")
print("Tracking URI:", mlflow.get_tracking_uri())

mlflow.set_experiment("customer-churn-02")
from mlflow.tracking import MlflowClient
client = MlflowClient()


d:\Documents\ds-ml-portfolio\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000


In [2]:
with open('./data/train.csv', 'r') as f:
    df = pd.read_csv(f)

In [3]:
df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
def get_num_and_cat_cols(df):
    numeric_cols = []
    categorical_cols = []

    for col in df.columns:
        if df[col].dtype in ['object', 'str'] or df[col].nunique() == 2:
            categorical_cols.append(col)
        elif df[col].dtype in ['int64', 'float64']:
            if df[col].nunique() < 6:
                categorical_cols.append(col)
            else:
                numeric_cols.append(col)
    
    return (numeric_cols, categorical_cols)

In [5]:
numeric_cols, categorical_cols = get_num_and_cat_cols(df)

print(f"Numeric Columns: {numeric_cols}")
print(f"Categorical Columns: {categorical_cols}")

Numeric Columns: ['id', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


---
# 3. Dataset split

In [19]:
df_copy = df.copy()
numeric_cols, categorical_cols = get_num_and_cat_cols(df_copy)


numeric_cols_ = numeric_cols.copy()
numeric_cols_.remove("id")
categorical_cols_ = categorical_cols.copy()
categorical_cols_.remove("Churn")
X = df_copy.drop("id", axis=1)
X = X.drop("Churn", axis=1)
X = pd.concat([
    X.select_dtypes([], ['object']),
    X.select_dtypes(['object']).apply(pd.Series.astype, dtype='category')
    ], axis=1)


y = df_copy["Churn"].map({"Yes": 1, "No": 0})

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# 4. Feature engineering

In [101]:
class NormalizationTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.scaler = MinMaxScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns])
        return self
    
    def transform(self, X):
        X = X.copy()
        X_norm = X[self.columns]

        X_norm = pd.DataFrame(
            self.scaler.transform(X_norm),
            columns=[f"{col}Norm" for col in self.columns],
            index=X.index
        )

        X = pd.concat([X, X_norm], axis=1)
        
        return X

class StandarizationTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        if not set(self.columns).issubset(X.columns):
            raise ValueError(f"At least one column from {self.columns} not in X")

        self.scaler.fit(X[self.columns])
        return self
    
    def transform(self, X):
        X = X.copy()
        X_std = X[self.columns]

        X_std = pd.DataFrame(
            self.scaler.transform(X_std),
            columns=[f"{col}Std" for col in self.columns],
            index=X.index
        )

        X = pd.concat([X, X_std], axis=1)
        
        return X

class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        if not set(self.columns).issubset(X.columns):
            raise ValueError(f"At least one column from {self.columns} not in X")

        return self
    
    def transform(self, X):
        X = X.copy()

        for col in self.columns:
            if 0 in X[col].unique():
                X[f'{col}Log'] = np.log(X[col].clip(lower=1e-9))
            else:
                X[f'{col}Log'] = np.log(X[col])
        
        return X

class CatFractionEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.map_ = {}

    def fit(self, X, y=None):
        if not set(self.columns).issubset(X.columns):
            raise ValueError(f"At least one column from {self.columns} not in X")

        l = len(X)

        for col in self.columns:
            self.map_[col] = X[col].value_counts() / l

        return self
    
    def transform(self, X):
        X = X.copy()

        for col in self.columns:
            X[f'{col}Frac'] = X[col].map(self.map_[col].fillna(0)).astype(float)

        return X

class BinTranformer(BaseEstimator, TransformerMixin):
    def __init__(self, col_name, bins, labels):
        self.col_name = col_name
        self.bins = bins
        self.labels = labels

    def fit(self, X, y=None):
        if self.col_name not in X.columns:
            raise ValueError(f"Column {self.col_name} not in DataFrame")
        
        if self.labels is not None and len(self.labels) != self.bins:
            raise ValueError("Número de labels deve ser igual ao número de bins")
        return self
    
    def transform(self, X):
        X = X.copy()

        X[f'{self.col_name}Bins'] = pd.cut(X[f'{self.col_name}'], bins=self.bins, labels=self.labels).astype("category")

        return X
    
class RatioTranformer(BaseEstimator, TransformerMixin):
    def __init__(self, numerator, denominator):
        self.numerator = numerator
        self.denominator = denominator

    def fit(self, X, y=None):
        if self.numerator not in X.columns:
            raise ValueError(f"Column {self.numerator} not in DataFrame")
        
        if self.denominator not in X.columns:
            raise ValueError(f"Column {self.denominator} not in DataFrame")
        return self

    def transform(self, X):
        X = X.copy()

        X[f"{self.numerator}And{self.denominator}Ratio"] = X[self.numerator] / X[self.denominator]
        return X
    
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.map_ = {}
        self.global_mean = None

    def fit(self, X, y=None):
        if not set(self.columns).issubset(X.columns):
            raise ValueError(f"At least one column from {self.columns} not in X")

        df = X.copy()
        df["Churn"] = y.map({"Yes": 1, "No": 0})
        self.global_mean = df["Churn"].mean()

        for col in self.columns:
            self.map_[col] = (df.groupby(col)["Churn"].mean().to_dict())

        return self
    
    def transform(self, X):
        X = X.copy()

        for col in self.columns:
            X[f"{col}ChurnMean"] = (X[col].astype("object").map(self.map_[col]).fillna(self.global_mean))

        return X
    
class AditionalInternetServiceAggregator(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.services_list = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

    def fit(self, X, y=None):
        if not set(self.services_list).issubset(X.columns):
            raise ValueError(f"At least one column from {self.services_list} not in X")

        return self
    
    def transform(self, X):
        X = X.copy()
        X[f'AdditionalInternetServicesCount'] = (X[self.services_list] == 'Yes').sum(axis=1)

        return X
    
class BothServicesAggregator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if "PhoneService" not in X.columns:
            raise ValueError(f"Column {"PhoneService"} not in DataFrame")
        if "InternetService" not in X.columns:
            raise ValueError(f"Column {"InternetService"} not in DataFrame")

        return self
    
    def transform(self, X):
        X = X.copy()
        X['BothServices'] = ((X['PhoneService'] == "Yes") & (X['InternetService'] == "Yes")).map({True: 'Yes', False: 'No'}).astype("category")

        return X

In [ ]:
test = BinTranformer("tenure", 3, ["low", "medium", "high"])
# test.fit(X_train, y_train)
# test.transform(X_train).dtypes

Pipeline([
    ()
])

TypeError: 'BinTranformer' object is not reversible

TypeError: 'BinTranformer' object is not reversible

Pipeline(steps=BinTranformer(bins=3, col_name='tenure',
                             labels=['low', 'medium', 'high']))

In [106]:
feature_engineering_pipeline = Pipeline([
    # ("norm", NormalizationTransformer(numeric_cols_)),
    # ("std", StandarizationTransformer(numeric_cols_)),
    # ("log", LogTransformer(numeric_cols_)),
    # ("cat_frac", CatFractionEncoder(categorical_cols_)),
    # ("churn_encoder", TargetEncoder(categorical_cols_)),
    # ("tenure_bin", BinTranformer("tenure", 3, ["low", "medium", "high"])),
    # ("MonthlyCharges_bin", BinTranformer("MonthlyCharges", 3, ["low", "medium", "high"])),
    # ("TotalCharges_bin", BinTranformer("TotalCharges", 3, ["low", "medium", "high"])),
    ("MonthlyCharges_Tenure_ratio", RatioTranformer("MonthlyCharges", "tenure")),
    ("TotalCharges_Tenure_ratio", RatioTranformer("TotalCharges", "tenure")),
    # ("Charges_ratio", RatioTranformer("TotalCharges", "MonthlyCharges")),
    # ("aditional_internet_service", AditionalInternetServiceAggregator()),
    # ("both_services", BothServicesAggregator()),
])

# 5. Modeling

## XGBoost with RandomSearchCV

In [107]:
model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            enable_categorical=True,
            scale_pos_weight = 460000 / 133000,
            n_jobs=1
        )

full_pipeline = Pipeline([
    ("feature_engineering", feature_engineering_pipeline),
    ("model", model)
])

with mlflow.start_run(run_name="random_search_trial") as run:
    mlflow.set_tag("tuning_type", "random-search")
    run_id = run.info.run_id

    param_grid = {
        'model__n_estimators': [200, 400, 600, 800, 1000],
        'model__max_depth': [2, 4, 8, 16, 32],
        'model__learning_rate': [0.01, 0.1, 0.3],
        'model__booster': ['gbtree'],
        'model__subsample': [0.8, 1],
        'model__colsample_bytree': [0.8, 1],
        'model__min_child_weight': [1, 3, 5, 7],
        'model__gamma': [0, 0.1, 0.3, 1],
        'model__reg_alpha': [0, 0.1, 1, 10],
        'model__reg_lambda': [0, 0.5, 1, 5, 10],

    }

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    search = RandomizedSearchCV(
        estimator=full_pipeline,
        param_distributions=param_grid,
        n_iter=1,
        scoring="roc_auc",
        cv=cv,
        verbose=3,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    print("Best params:")
    print(search.best_params_)

    print("Best mean AUC:")
    print(search.best_score_)
    
#     mlflow.log_param("n_estimators", search.best_params_['model__n_estimators'])
#     mlflow.log_param("max_depth", search.best_params_['model__max_depth'])
#     mlflow.log_param("learning_rate", search.best_params_['model__learning_rate'])
#     mlflow.log_param("booster", search.best_params_['model__booster'])
#     mlflow.log_param("subsample", search.best_params_['model__subsample'])
#     mlflow.log_param("colsample_bytree", search.best_params_['model__colsample_bytree'])
#     mlflow.log_metric("roc_auc", search.score(X_val, y_val))
#     mlflow.sklearn.log_model(search.best_estimator_, name='grid_xgb_model', input_example=X_train.iloc[[0]], registered_model_name="customer-churn",)

# versions = client.search_model_versions(
#     f"name='customer-churn' and run_id='{run_id}'"
# )
# version = versions[0].version

# client.set_registered_model_alias(
#     name="customer-churn",
#     alias="candidate",
#     version=version
# )

Fitting 3 folds for each of 1 candidates, totalling 3 fits


d:\Documents\ds-ml-portfolio\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan]
  warnings.warn(


Best params:
{'model__subsample': 0.8, 'model__reg_lambda': 1, 'model__reg_alpha': 1, 'model__n_estimators': 1000, 'model__min_child_weight': 5, 'model__max_depth': 2, 'model__learning_rate': 0.3, 'model__gamma': 1, 'model__colsample_bytree': 0.8, 'model__booster': 'gbtree'}
Best mean AUC:
nan
🏃 View run random_search_trial at: http://localhost:5000/#/experiments/2/runs/dcea19d4bed9478ba7fbd3cf33665c61
🧪 View experiment at: http://localhost:5000/#/experiments/2


In [ ]:
def promote_to_stg(version):
    client.set_registered_model_alias(
        name="customer-churn",
        alias="stg",
        version=version
    )


prd_version = None
stg_version = None

try:
    prd_version = client.get_model_version_by_alias("customer-churn", alias="prd")
    prd_run_id = prd_version.run_id
    prd_run = client.get_run(prd_run_id)

    prd_auc = prd_run.data.metrics["roc_auc"]
except Exception as e:
    print(e)

if prd_version is None:
    try:
        stg_version = client.get_model_version_by_alias("customer-churn", alias="stg")
        stg_run_id = stg_version.run_id
        stg_run = client.get_run(stg_run_id)

        stg_auc = stg_run.data.metrics["roc_auc"]
    except Exception as e:
        print(e)


candidate_version = client.get_model_version_by_alias("customer-churn", alias="candidate")
candidate_run_id = candidate_version.run_id
candidate_run = client.get_run(candidate_run_id)

candidate_auc = candidate_run.data.metrics["roc_auc"]

if (stg_version is None and prd_version is None) or (prd_version is None and candidate_auc > stg_auc) or (prd_version is not None and candidate_auc > prd_auc):
    print("Promoting to stage")
    promote_to_stg(version)

Promoting to stage
